# Fix Backslash Image Paths in Parquet

Some parquet files created on Windows contain ShareGPT-4o image entries stored as  
`{"bytes": None, "path": "data/ShareGPT-4o/mnt/.../image\\41477.jpg"}`.  
The single backslash (chr 92) is the Windows `os.path.join` separator baked into the
stored path string.  On Linux it is **not** a directory separator, so
`PIL.Image.open(path)` raises `FileNotFoundError` when the training run loads the dataset.

This notebook:
1. Scans the target parquet and reports affected rows
2. Replaces every `\\` → `/` in stored `path` strings
3. Writes the fixed parquet back in-place (original backed up as `.parquet.bak`)

In [1]:
import os
from pathlib import Path

def _repo_root() -> Path:
    p = Path.cwd().resolve()
    if (p / "data").is_dir():
        return p
    if (p.parent / "data").is_dir():
        return p.parent
    return p

REPO_ROOT = _repo_root()
os.chdir(REPO_ROOT)
print("Repo root:", REPO_ROOT)

Repo root: /root/autodl-tmp/Qwen3-SmVL


In [2]:
import pyarrow as pa
import pyarrow.parquet as pq

PARQUET_PATH = REPO_ROOT / "data" / "the_cauldron_ZH_filtered_merged" / "the_cauldron_ZH_dsV32_filtered_merged.parquet"

print(f"Target : {PARQUET_PATH}")
print(f"Size   : {PARQUET_PATH.stat().st_size / 1024**3:.2f} GB")

Target : /root/autodl-tmp/Qwen3-SmVL/data/the_cauldron_ZH_filtered_merged/the_cauldron_ZH_dsV32_filtered_merged.parquet
Size   : 13.60 GB


## 1. Audit — count affected rows

In [ ]:
BS = chr(92)   # backslash character — avoids escaping confusion throughout the notebook

pf = pq.ParquetFile(PARQUET_PATH)

total_rows   = 0
path_only    = 0   # bytes is None
backslash    = 0   # bytes is None AND path contains a backslash
sample_paths = []

for batch in pf.iter_batches(batch_size=2000, columns=["images"]):
    for row in batch.column("images").to_pylist():
        total_rows += 1
        for img in (row or []):
            if not isinstance(img, dict):
                continue
            if img.get("bytes") is None:
                path_only += 1
                p = img.get("path") or ""
                if BS in p:
                    backslash += 1
                    if len(sample_paths) < 5:
                        sample_paths.append(p)

print(f"Total rows          : {total_rows:,}")
print(f"Path-only (no bytes): {path_only:,}")
print(f"Backslash paths     : {backslash:,}")
print()
print("Sample affected paths:")
for p in sample_paths:
    print(" ", repr(p))

## 2. Fix function

In [ ]:
def fix_image_paths_in_table(table: pa.Table) -> tuple[pa.Table, int]:
    """
    Walk the images column (list<struct<bytes: binary, path: string>>),
    replace every backslash (chr 92) in stored path strings with a forward slash,
    and return (fixed_table, n_fixed) where n_fixed is the count of changed entries.
    """
    BS = chr(92)
    images_raw = table.column("images").to_pylist()
    n_fixed = 0

    fixed_rows = []
    for row in images_raw:
        fixed_row = []
        for img in (row or []):
            if isinstance(img, dict):
                p = img.get("path") or ""
                if BS in p:
                    img = {**img, "path": p.replace(BS, "/")}
                    n_fixed += 1
            fixed_row.append(img)
        fixed_rows.append(fixed_row)

    # Reconstruct the column preserving the original Arrow type
    img_type   = table.schema.field("images").type
    fixed_col  = pa.array(fixed_rows, type=img_type)
    fixed_table = table.set_column(
        table.schema.get_field_index("images"),
        "images",
        fixed_col,
    )
    return fixed_table, n_fixed


# ── Sanity-check on a small slice ────────────────────────────────────────────
sample_table = pq.read_table(PARQUET_PATH).slice(0, 5)
fixed_sample, n = fix_image_paths_in_table(sample_table)

print(f"Fixed {n} path(s) in sample slice.")
print()
for i, row in enumerate(fixed_sample.column("images").to_pylist()):
    for img in (row or []):
        if isinstance(img, dict):
            has_bytes = img.get("bytes") is not None
            print(f"  Row {i} | bytes: {str(has_bytes):5s} | path: {repr(img.get('path'))}")

## 3. Apply fix to the full parquet and overwrite

In [ ]:
import shutil

# ── Read ─────────────────────────────────────────────────────────────────────
print("Reading full parquet…")
table = pq.read_table(PARQUET_PATH)
print(f"  {len(table):,} rows loaded")

# ── Fix ──────────────────────────────────────────────────────────────────────
print("Fixing backslash paths…")
fixed_table, n_fixed = fix_image_paths_in_table(table)
print(f"  {n_fixed:,} image path(s) fixed")

# ── Backup original (once) ───────────────────────────────────────────────────
backup_path = PARQUET_PATH.with_suffix(".parquet.bak")
if not backup_path.exists():
    shutil.copy2(PARQUET_PATH, backup_path)
    print(f"  Backup → {backup_path.name}")
else:
    print(f"  Backup already exists, skipping ({backup_path.name})")

# ── Write back ───────────────────────────────────────────────────────────────
print("Writing fixed parquet…")
pq.write_table(fixed_table, PARQUET_PATH, compression="snappy")
new_size = PARQUET_PATH.stat().st_size / 1024**3
print(f"  Saved → {PARQUET_PATH.name}  ({new_size:.2f} GB)")

## 4. Verify — re-scan the saved file

In [3]:
BS = chr(92)
pf2 = pq.ParquetFile(PARQUET_PATH)

remaining_backslash = 0
remaining_path_only = 0
sample_fixed = []

for batch in pf2.iter_batches(batch_size=2000, columns=["images"]):
    for row in batch.column("images").to_pylist():
        for img in (row or []):
            if not isinstance(img, dict):
                continue
            if img.get("bytes") is None:
                remaining_path_only += 1
                p = img.get("path") or ""
                if BS in p:
                    remaining_backslash += 1
                elif len(sample_fixed) < 5:
                    sample_fixed.append(p)

print(f"Remaining backslash paths : {remaining_backslash}  (should be 0)")
print(f"Remaining path-only rows  : {remaining_path_only:,}")
print()
print("Sample fixed paths (forward slashes only):")
for p in sample_fixed:
    print(" ", repr(p))

Remaining backslash paths : 0  (should be 0)
Remaining path-only rows  : 34,482

Sample fixed paths (forward slashes only):
  'data/ShareGPT-4o/mnt/petrelfs/wangwenhai/workspace_cef/4o/image/10765.jpg'
  'data/ShareGPT-4o/mnt/petrelfs/wangwenhai/workspace_cef/4o/image/21241.jpg'
  'data/ShareGPT-4o/mnt/petrelfs/wangwenhai/workspace_cef/4o/image/9421.jpg'
  'data/ShareGPT-4o/mnt/petrelfs/wangwenhai/workspace_cef/4o/image/54826.jpg'
  'data/ShareGPT-4o/mnt/petrelfs/wangwenhai/workspace_cef/4o/image/52446.jpg'
